<h3>HWP와 HWPX 실습 준비</h3>
<p><strong>개념</strong> HWPX는 ZIP 압축 폴더처럼 여러 XML 파일을 한 문서 파일 안에 담습니다. 이 실습은 제공된 최소 HWPX의 내부 XML을 확인하며, 레거시 HWP 본문 추출이나 한컴오피스 호환성을 검증하지 않습니다.</p>

In [1]:
# sys는 현재 노트북 커널의 Python 정보를 제공합니다.
# 버전 번호를 출력해 실습 기준인 Python 3.12 환경인지 확인합니다.
import sys
print("Python:", sys.version.split()[0])


Python: 3.12.13


<h3>실행 위치와 제공 파일 확인</h3>
<p>Ch10의 <code>fixtures</code> 폴더에서 HWPX 표식 파일과 레거시 HWP 표식 파일을 찾습니다. 두 파일은 내부가 ZIP인지 확인한 뒤 읽는 방법을 나누는 연습용이며 실제 업무 문서를 대신하지 않습니다.</p>

In [2]:
# HWPX의 ZIP과 XML 구조를 읽을 표준 라이브러리를 준비합니다.
# 상위 폴더를 순서대로 확인해 Ch10 실습 폴더를 찾습니다.
from pathlib import Path
from xml.etree import ElementTree as ET
import zipfile

lab = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    possible = candidate / "labs/part3/ch10-document-processing"
    if possible.is_dir():
        lab = possible
        break
if lab is None:
    raise FileNotFoundError("저장소 안에서 노트북을 실행하세요.")
hwpx = lab / "fixtures/minimal-marker.hwpx"
legacy = lab / "fixtures/legacy-signature.hwp"
for path in (hwpx, legacy):
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.name, path.stat().st_size, "bytes")


minimal-marker.hwpx 470 bytes
legacy-signature.hwp 23 bytes


<h3>ZIP 방식으로 묶인 내부 파일 관찰</h3>
<p><code>is_zipfile()</code>은 파일이 ZIP 구조인지 확인합니다. <code>namelist()</code>는 압축 안에 들어 있는 파일 이름 목록을 반환합니다.</p>

In [3]:
# 먼저 HWPX 표식 파일이 ZIP인지 확인하고 내부 항목을 읽습니다.
# with 블록을 벗어나면 archive 객체가 파일을 자동으로 닫습니다.
assert zipfile.is_zipfile(hwpx)
with zipfile.ZipFile(hwpx) as archive:
    entries = archive.namelist()
print("ZIP 항목 수:", len(entries))
print("앞의 항목:", entries[:5])


ZIP 항목 수: 2
앞의 항목: ['Contents/content.hpf', 'Contents/section0.xml']


<h3>본문 XML만 선택</h3>
<p>전체 ZIP 항목에서 <code>Contents/section</code>으로 시작하고 <code>.xml</code>로 끝나는 항목만 고릅니다. 필터 조건을 바꾸면 선택 결과가 달라집니다.</p>

In [4]:
# 전체 항목을 순회하며 본문 section XML의 이름만 sections에 추가합니다.
# startswith()와 endswith()는 문자열의 시작과 끝 조건을 각각 검사합니다.
sections = []
for name in entries:
    is_section = name.startswith("Contents/section")
    is_xml = name.endswith(".xml")
    if is_section and is_xml:
        sections.append(name)
sections.sort()
print("전체 항목:", len(entries))
print("본문 XML:", sections)


전체 항목: 2
본문 XML: ['Contents/section0.xml']


<h3>XML 객체와 본문 값 관찰</h3>
<p><code>ET.fromstring()</code>은 XML 바이트를 요소 객체로 바꿉니다. 각 태그의 로컬 이름이 <code>t</code>인 요소만 골라 텍스트를 합칩니다.</p>

In [5]:
# 첫 본문 XML을 바이트로 읽고 XML 루트 요소를 만듭니다.
# iter()가 반환하는 모든 요소를 확인해 t 태그의 텍스트만 저장합니다.
with zipfile.ZipFile(hwpx) as archive:
    xml_bytes = archive.read(sections[0])
root = ET.fromstring(xml_bytes)
text_parts = []
for node in root.iter():
    local_name = node.tag.rsplit("}", 1)[-1]
    if local_name == "t":
        text_parts.append(node.text or "")
text = "".join(text_parts)
print("루트 태그:", root.tag)
print("추출 본문:", text)


루트 태그: {http://www.hancom.co.kr/hwpml/2011/paragraph}section
추출 본문: 한글 문서 처리 실습입니다.


<h3>선택 조건을 바꾸어 비교</h3>
<p>같은 ZIP 목록에서 선택 조건만 바꿉니다. 모든 XML을 선택한 결과와 본문 section XML만 선택한 결과의 개수를 비교합니다.</p>

In [6]:
# suffix 조건만 적용하면 설정 XML도 포함될 수 있습니다.
# 본문 경로 조건을 함께 적용한 sections와 개수를 비교합니다.
all_xml = []
for name in entries:
    if name.endswith(".xml"):
        all_xml.append(name)
print("모든 XML:", len(all_xml))
print("본문 section XML:", len(sections))
assert len(all_xml) >= len(sections)


모든 XML: 1
본문 section XML: 1


<h3>일부러 실패해 보기</h3>
<p><strong>Before 원인</strong> 레거시 HWP는 HWPX와 같은 ZIP 형식이 아닙니다. 확장자만 보고 ZIP으로 열면 <code>BadZipFile</code>이 발생합니다.</p>

In [7]:
# 레거시 HWP 표식 파일을 HWPX처럼 ZIP으로 여는 오류를 재현합니다.
# BadZipFile만 포착해 전용 HWP 파서가 필요한 경계를 확인합니다.
try:
    with zipfile.ZipFile(legacy) as archive:
        print(archive.namelist())
except zipfile.BadZipFile as error:
    print(f"caught: legacy HWP needs a dedicated parser ({error})")


caught: legacy HWP needs a dedicated parser (File is not a zip file)


<p><strong>After 수정</strong> 파일 앞부분과 ZIP 구조를 먼저 확인합니다. 레거시 HWP 표식은 미지원으로 분류하고, HWPX만 ZIP/XML 경로에서 읽습니다.</p>

In [8]:
# 이 표식의 OLE 시그니처는 레거시 HWP를 ZIP 처리에서 제외하는 기준입니다.
# 실제 업무에서는 전용 파서로 HWP 고유 구조까지 추가 검증해야 합니다.
ole_signature = bytes.fromhex("D0CF11E0A1B11AE1")
results = {}
for path in (legacy, hwpx):
    header = path.read_bytes()[:8]
    if header == ole_signature:
        results[path.suffix] = "전용 파서가 필요한 미지원 HWP"
        continue
    if not zipfile.is_zipfile(path):
        raise ValueError("HWPX ZIP 형식이 아닙니다.")
    with zipfile.ZipFile(path) as archive:
        xml_bytes = archive.read(sections[0])
    xml_root = ET.fromstring(xml_bytes)
    parts = []
    for node in xml_root.iter():
        if node.tag.rsplit("}", 1)[-1] == "t":
            parts.append(node.text or "")
    results[path.suffix] = "".join(parts)
assert results[".hwp"] == "전용 파서가 필요한 미지원 HWP"
assert results[".hwpx"] == text
print(results)


{'.hwp': '전용 파서가 필요한 미지원 HWP', '.hwpx': '한글 문서 처리 실습입니다.'}
